In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from itertools import product
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\SpentTimeModel\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/SpentTimeModel/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cpu')

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

In [5]:
BATCH_SIZE = 1024
EPOCHS = 50
VOCAB_SIZE = 601
TIME_SIZE = 301

run_models = 1

In [6]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [7]:
with open(data_folder + 'data_time_features.pkl', 'rb') as f:
    df_time_features = pickle.load(f)

In [8]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [9]:
list_seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
list_context = [5]
list_embedding_layer_dim = [32, 64]
list_num_neurons = [50]
list_num_layers = [3, 5]
list_dropout = [0, 0.3]
list_layernorm = [False, True]
list_set_covariables = [1, 2]

In [10]:
model_mask_name_total = "{}_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"
model_mask_name_46min = "{}_first46m_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"
model_mask_name_2min = "{}_last2m_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"

In [11]:
all_combinations = list(product(list_seasons, list_context, list_embedding_layer_dim, list_num_neurons, list_num_layers,
                                list_dropout, list_layernorm, list_set_covariables))

In [12]:
len(all_combinations)

160

In [13]:
for comb in all_combinations:
    season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, NUM_LAYERS, dropout_rate, layer_norm, key_covariables = comb

    print(comb)
    index_covariables = dict_set_covariables["index"][key_covariables]
    covariables = dict_set_covariables["covariables"][key_covariables]

    # ================== DADOS TREINO/DEV - MODELO TOTAL ==================
    df_ = df_time_features.query("season == @season & season_split == 'train'")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    df_ = df_time_features.query("season == @season & season_split == 'dev'")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    del df_
    # =======================================================================

    tp_dropout = "nodropout" if dropout_rate == 0 else "dropout" + str(dropout_rate).replace(".", "")

    time_model = SpentTimeModel(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        time_size = TIME_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation_function = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_total\
    .format(time_model.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    if results_filename in os.listdir(models_folder):
        print("Already fitted.")
        continue

    # ================== TREINO MODELO TOTAL ==================
    token_model, trnloss, devloss = train_spent_time_model(
        model = time_model,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.002,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = time_model,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        numerical_features = dev_num[:, index_covariables],
        names_numerical_features = covariables,
        num_repeats = 10,
        mask_data = dev_mask)

    print(df_importance)

    output = time_model(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables])
    output = output.masked_fill(trn_mask, -1e9)
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(time_model.state_dict(), models_folder + model_filename)

    dict_results = {
        "class_model": time_model.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(time_model),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    #######################################################################
    # ================== MODELO 46 MINUTOS (FINE-TUNING) ==================
    df_ = df_time_features.query("season == @season & season_split == 'train' & ind_last_2_minutes_game == 0")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    df_ = df_time_features.query("season == @season & season_split == 'dev' & ind_last_2_minutes_game == 0")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    del df_

    time_model_46m = SpentTimeModel(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        time_size = TIME_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation_function = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_46min\
    .format(time_model_46m.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    # ================== TREINO MODELO FIRST 46 MIN ==================
    # inicializa com pesos do modelo total
    time_model_46m.load_state_dict(time_model.state_dict())

    time_model_46m, trnloss, devloss = train_spent_time_model(
        model = time_model_46m,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.001,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = time_model_46m,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        numerical_features = dev_num[:, index_covariables],
        names_numerical_features = covariables,
        num_repeats = 10,
        mask_data = dev_mask)

    print(df_importance)

    output = time_model_46m(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables])
    output = output.masked_fill(trn_mask, -1e9)    
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(time_model_46m.state_dict(), models_folder + model_filename)
    dict_results = {
        "class_model": time_model_46m.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(time_model_46m),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    #######################################################################
    # ================== MODELO  2 MINUTOS (FINE-TUNING) ==================
    df_ = df_time_features.query("season == @season & season_split == 'train' & ind_last_2_minutes_game == 1")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    df_ = df_time_features.query("season == @season & season_split == 'dev' & ind_last_2_minutes_game == 1")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    del df_

    time_model_2m = SpentTimeModel(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        time_size = TIME_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation_function = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_2min\
    .format(time_model_2m.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    # ================== TREINO MODELO LAST 2 MIN ==================
    # inicializa com pesos do modelo total
    time_model_2m.load_state_dict(time_model.state_dict())

    time_model_2m, trnloss, devloss = train_num_ngram_model(
        model = time_model_2m,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.001,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
    model = time_model_2m,
    target = dev_y,
    tokens = dev_tkn[:, -CONTEXT_SIZE:],
    numerical_features = dev_num[:, index_covariables],
    names_numerical_features = covariables,
    num_repeats = 10,
    mask_data = dev_mask)

    print(df_importance)

    output = time_model_2m(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables])
    output = output.masked_fill(trn_mask, -1e9)
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(time_model_2m.state_dict(), models_folder + model_filename)
    dict_results = {
        "class_model": time_model_2m.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(time_model_2m),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    clear_output(wait = True)



('2022-23', 5, 64, 50, 5, 0.3, True, 2)
Baseline model loss:  2.3685281 | Val Loss: 2.36854 | LR: 4.88e-07
                            feature  count    mean     std     min     25%     50%     75%     max  baseline     perc   percacm
4                           token_1   10.0  2.2954  0.0119  4.6453  4.6570  4.6614  4.6734  4.6809    2.3685  51.6358   51.6358
3                           token_2   10.0  0.8437  0.0058  3.2049  3.2078  3.2121  3.2159  3.2212    2.3685  18.9805   70.6163
19                  ind_last_minute   10.0  0.6599  0.0030  3.0239  3.0257  3.0286  3.0312  3.0319    2.3685  14.8444   85.4608
2                           token_3   10.0  0.1380  0.0014  2.5033  2.5059  2.5070  2.5073  2.5083    2.3685   3.1044   88.5652
20              ind_last_20_seconds   10.0  0.1347  0.0009  2.5018  2.5027  2.5034  2.5036  2.5045    2.3685   3.0311   91.5963
22               ind_last_5_seconds   10.0  0.1242  0.0011  2.4912  2.4921  2.4925  2.4932  2.4946    2.3685   2.7940   94.39